# 건축 법규 MCP 서버 — KDS 구조 검토

건축공학 도메인에 특화된 MCP 서버를 직접 설계합니다.

## 목표
KDS 41 17 00 기준에 따른 RC 부재 구조 검토 MCP 서버를 구축합니다.

## 단계별 빌드업
- **v1**: 도구만 (휨 검토) — `@mcp.tool()` + Inspector 테스트
- **v2**: 도구 + 리소스 (재료 물성치) — `@mcp.resource()` 추가
- **v3**: 종합 (+ 프롬프트) — `@mcp.prompt()` 추가 + 클라이언트 통합

In [ ]:
# ── Setup ──────────────────────────────────────────────
import json
import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Structural Engineering MCP Server")

## v1: 도구 정의 — 휨 강도 검토

KDS 41 17 00에 따른 RC 보의 휨 강도 검토 도구를 만듭니다.

In [ ]:
@mcp.tool()
def check_flexural_strength(
    b: float,
    d: float,
    As: float,
    fck: float,
    fy: float,
    Mu: float
) -> str:
    """RC 보의 휨 강도를 KDS 41 17 00 기준으로 검토합니다.

    Args:
        b: 보 폭 (mm)
        d: 유효 깊이 (mm)
        As: 인장 철근 단면적 (mm2)
        fck: 콘크리트 설계기준 압축강도 (MPa)
        fy: 철근 항복강도 (MPa)
        Mu: 소요 휨모멘트 (kN.m)
    """
    # 등가 직사각형 응력블록 깊이
    a = As * fy / (0.85 * fck * b)

    # 공칭 휨강도
    Mn = As * fy * (d - a / 2) / 1e6  # kN.m

    # 설계 휨강도 (phi = 0.85 for flexure)
    phi = 0.85
    phi_Mn = phi * Mn

    # 철근비 검토
    rho = As / (b * d)
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)

    # 판정
    flexure_ok = phi_Mn >= Mu
    rho_ok = rho >= rho_min

    result = {
        "section": f"{b}x{d} mm",
        "As": f"{As:.1f} mm2",
        "a": f"{a:.1f} mm",
        "Mn": f"{Mn:.1f} kN.m",
        "phi_Mn": f"{phi_Mn:.1f} kN.m",
        "Mu": f"{Mu:.1f} kN.m",
        "flexure_check": "OK" if flexure_ok else "NG",
        "DCR": f"{Mu / phi_Mn:.3f}",
        "rho": f"{rho:.5f}",
        "rho_min": f"{rho_min:.5f}",
        "rho_check": "OK" if rho_ok else "NG (최소 철근비 미달)"
    }
    return json.dumps(result, indent=2, ensure_ascii=False)


@mcp.tool()
def check_shear_strength(
    b: float,
    d: float,
    fck: float,
    Av: float,
    s: float,
    fy: float,
    Vu: float
) -> str:
    """RC 보의 전단 강도를 KDS 41 17 00 기준으로 검토합니다.

    Args:
        b: 보 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준 압축강도 (MPa)
        Av: 전단 보강근 단면적 (mm2)
        s: 전단 보강근 간격 (mm)
        fy: 전단 보강근 항복강도 (MPa)
        Vu: 소요 전단력 (kN)
    """
    # 콘크리트 전단강도
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000  # kN

    # 철근 전단강도
    Vs = Av * fy * d / s / 1000  # kN

    # 공칭 전단강도
    Vn = Vc + Vs

    # 설계 전단강도 (phi = 0.75)
    phi = 0.75
    phi_Vn = phi * Vn

    # 판정
    shear_ok = phi_Vn >= Vu

    result = {
        "Vc": f"{Vc:.1f} kN",
        "Vs": f"{Vs:.1f} kN",
        "Vn": f"{Vn:.1f} kN",
        "phi_Vn": f"{phi_Vn:.1f} kN",
        "Vu": f"{Vu:.1f} kN",
        "shear_check": "OK" if shear_ok else "NG",
        "DCR": f"{Vu / phi_Vn:.3f}"
    }
    return json.dumps(result, indent=2, ensure_ascii=False)


print("v1 완료: 2개 도구 (휨 + 전단) 등록")

In [ ]:
# v1 테스트: 휨 강도 검토
import asyncio

async def test_v1():
    # 300x600 보, 4-D25 (As=1,963mm2), fck=27, fy=400, Mu=200kN.m
    result = await mcp.call_tool("check_flexural_strength", {
        "b": 300, "d": 540, "As": 1963, "fck": 27, "fy": 400, "Mu": 200
    })
    print("=== 휨 강도 검토 ===")
    print(result)

    # 전단 강도 검토
    result2 = await mcp.call_tool("check_shear_strength", {
        "b": 300, "d": 540, "fck": 27, "Av": 142, "s": 200, "fy": 400, "Vu": 150
    })
    print("\n=== 전단 강도 검토 ===")
    print(result2)

await test_v1()

## v2: 리소스 추가 — 재료 물성치 + KDS 기준

도구에 더해 **리소스**를 추가합니다. 재료 물성치와 설계 기준을 리소스로 제공합니다.

In [ ]:
# 리소스 1: KDS 41 17 00 기준 요약
@mcp.resource("kds://41-17-00/summary")
def get_kds_summary() -> str:
    """KDS 41 17 00 (콘크리트구조 설계기준) 주요 내용을 반환합니다."""
    return json.dumps({
        "title": "KDS 41 17 00 콘크리트구조 설계기준",
        "flexure": {
            "phi": 0.85,
            "rho_min": "max(0.25*sqrt(fck)/fy, 1.4/fy)",
            "equivalent_stress_block": "a = As*fy / (0.85*fck*b)"
        },
        "shear": {
            "phi": 0.75,
            "Vc": "(1/6)*sqrt(fck)*b*d",
            "Vs": "Av*fy*d/s"
        }
    }, indent=2, ensure_ascii=False)


# 리소스 2: 콘크리트 물성치 테이블
@mcp.resource("data://materials/concrete-table")
def get_concrete_table() -> str:
    """콘크리트 강도별 물성치 테이블을 반환합니다."""
    return json.dumps({
        "C24": {"fck": 24, "Ec": 25742, "fr": 3.10},
        "C27": {"fck": 27, "Ec": 26871, "fr": 3.29},
        "C30": {"fck": 30, "Ec": 27924, "fr": 3.46},
        "C35": {"fck": 35, "Ec": 29388, "fr": 3.74},
        "C40": {"fck": 40, "Ec": 30722, "fr": 4.00},
        "note": "Ec = 8500 * fck^(1/3), fr = 0.63*sqrt(fck)"
    }, indent=2, ensure_ascii=False)


# 리소스 3: 철근 규격 테이블
@mcp.resource("data://materials/rebar-table")
def get_rebar_table() -> str:
    """철근 규격별 단면적 테이블을 반환합니다."""
    return json.dumps({
        "D10": {"db": 9.53, "As": 71.3},
        "D13": {"db": 12.7, "As": 126.7},
        "D16": {"db": 15.9, "As": 198.6},
        "D19": {"db": 19.1, "As": 286.5},
        "D22": {"db": 22.2, "As": 387.1},
        "D25": {"db": 25.4, "As": 506.7},
        "D29": {"db": 28.6, "As": 642.4},
        "D32": {"db": 31.8, "As": 794.2},
        "note": "db: 공칭 직경(mm), As: 공칭 단면적(mm2)"
    }, indent=2, ensure_ascii=False)


print("v2 완료: 3개 리소스 추가 (KDS 요약 + 콘크리트 + 철근)")

In [ ]:
# v2 테스트: 리소스 읽기
async def test_v2():
    resources = await mcp.list_resources()
    print(f"등록된 리소스 ({len(resources)}개):")
    for r in resources:
        print(f"  - {r.uri}")

    print("\n=== 철근 규격 테이블 ===")
    content = await mcp.read_resource("data://materials/rebar-table")
    print(content)

await test_v2()

## v3: 프롬프트 추가 — 구조 검토 템플릿

프롬프트를 추가하여 MCP 서버의 3대 기능을 모두 갖춥니다.

In [ ]:
@mcp.prompt()
def structural_review(
    member_type: str,
    b: float,
    d: float,
    fck: float,
    fy: float
) -> str:
    """RC 부재 구조 검토 프롬프트입니다.

    Args:
        member_type: 부재 종류 (beam, column)
        b: 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 강도 (MPa)
        fy: 철근 강도 (MPa)
    """
    return f"""당신은 건축구조 전문가입니다. KDS 41 17 00에 따라 다음 RC {member_type}를 검토해주세요.

부재: {member_type}, b={b}mm, d={d}mm, fck={fck}MPa, fy={fy}MPa

검토 항목:
1. 최소/최대 철근비 → check_flexural_strength 도구 사용
2. 휨 강도 → check_flexural_strength 도구 사용
3. 전단 강도 → check_shear_strength 도구 사용
4. 종합 판정

필요한 재료 데이터는 리소스에서 조회하세요."""


print("v3 완료: 프롬프트 추가 — MCP 서버 3대 기능 완성!")

In [ ]:
# v3 최종 테스트: 전체 기능 확인
async def test_v3():
    tools = await mcp.list_tools()
    resources = await mcp.list_resources()
    prompts = await mcp.list_prompts()

    print(f"Tools:     {len(tools)}개")
    print(f"Resources: {len(resources)}개")
    print(f"Prompts:   {len(prompts)}개")
    print("\n=== Structural Engineering MCP Server ===")
    print("\nTools:")
    for t in tools:
        print(f"  - {t.name}")
    print("\nResources:")
    for r in resources:
        print(f"  - {r.uri}")
    print("\nPrompts:")
    for p in prompts:
        print(f"  - {p.name}")

await test_v3()

## 도전 과제

1. **server.py로 내보내기** → `mcp dev server.py`로 Inspector 테스트
2. **Claude Desktop 연결** → `mcp install server.py`
3. **Claude Code 연결** → `claude mcp add structural -- python server.py`
4. Claude에게 "이 보의 휨 강도를 KDS 기준으로 검토해줘"라고 요청
5. (심화) 기둥 축력-휨 상호작용 도구 추가